"""
-------------------------------------------------------------------------------
PANORAMA BATCH STITCHER
-------------------------------------------------------------------------------
Batch process photomosaic construction.
Author: Brian Willis<br>
AI programming support: Gemini 3 pro<br>
Contact: Willis77019@gmail.com<br>
License:  MIT License (unrestricted: Use at your own risk)<br>

DESCRIPTION:
This script automates the stitching of multiple panoramic mosaics from a single
folder of images. It will stitch photomosaics constructed from a single row of photos or multiple rows of photos. It supports .JPG and Nikon RAW (.NEF) files (extendable to others).

PREREQUISITES:
1. Hugin must be installed.
   https://hugin.sourceforge.io/
   - You must point the script to the 'bin' directory containing "align_image_stack.exe".
   - Example Path: r"C:\Program Files\Hugin\bin"
2. Python Libraries:
   - pip install rawpy imageio exifread

IT USES THE HUGIN TOOLCHAIN:
1. pto_gen:         Creates the project file.
2. cpfind:          Finds matching control points in overlapping images (multi-row supported).
3. autooptimiser:   Calculates geometry, warp, and photometric alignment.
4. hugin_executor:  Renders the final high-resolution mosaic.

SHOOTING & SETUP:
- Optimized for ROTATIONAL stitching (standing in one spot, panning/tilting).
- LINEAR SCANS: If you walked along an outcrop (linear translation), this script 
  may struggle with parallax. For linear scans, consider Microsoft ICE or Meshroom.
- CLEAN DATA: The script groups images by timestamps. If you have rapid-burst 
  sequences (HDR brackets or sports mode) mixed in, the grouping logic will fail. 
  Please separate panorama sets into their own folder before processing.
- SHOOTING PATTERN: Multi-row photomosaics are best collected by a Gigapan robot 
  for a uniform distribution. This script is made for those who wish to make handheld 
  "Gigapan-like" mosaics. 
  Best practice:
  1. Ensure moderate overlap of images (30-40%).
  2. Shoot in a snake pattern (Left->Right, Tilt Up, Right->Left).
  3. Use a long focal length lens (preferably 85mm+ or 200mm) for flatter perspective.

RUN MODES (Set in Configuration Section):
1. Auto-Detect (Default):
   - Uses TIME_THRESHOLD (e.g., 20.0 seconds) to detect when one mosaic ends 
     and the next begins.
   - Requirement: You must pause for >20s between shooting different outcrops.
2. Force Mode (FORCE_ALL_ONE_PANO = True):
   - Treats the entire folder as a single giant mosaic.
   - Use this for very large/long datasets where you might have paused during shooting.

PERFORMANCE NOTE (The "Go To Bed" Rule):
This process is computationally intensive. 
- Phase 1 (Matching) creates many "tilted" single images in the output folder. 
  DO NOT PANIC. These are temporary intermediate files.
- Phase 2 (Rendering) compiles them into the final mosaic. 
- For large field seasons, it is recommended to run this script overnight.
-------------------------------------------------------------------------------
"""

In [2]:
"""
-------------------------------------------------------------------------------
PANORAMA PANNER (UNIVERSAL BATCH ENGINE)
-------------------------------------------------------------------------------
Author:       Brian Willis
Assisted by:  Gemini AI
License:      MIT License 

DESCRIPTION:
The Batch version of the "Raw Binary" engine.
1. Converts each image (JPG, TIF, PNG, RAW) to a raw binary stream on the SSD.
2. Memory-maps the stream to render the video.
3. Automatically cleans up the temp file before starting the next image.

PREREQUISITES:
- pip install imageio[ffmpeg] numpy opencv-python
-------------------------------------------------------------------------------
"""

import os
import sys
import subprocess
import re
import time
import glob
import tkinter as tk
from tkinter import filedialog

# === LIBRARIES ===
try:
    import imageio_ffmpeg
    import numpy as np
    import cv2
except ImportError:
    print("Missing libraries! Please run:")
    print("pip install imageio[ffmpeg] numpy opencv-python")
    sys.exit()

# === CONFIGURATION ===
VIDEO_WIDTH = 3840   # 4K UHD
VIDEO_HEIGHT = 2160
FPS = 30
DURATION = 20.0      # Duration per video in seconds

# ==============================

def get_image_dims(ffmpeg_exe, file_path):
    """Probes image dimensions using FFmpeg (Zero RAM)."""
    try:
        cmd = [ffmpeg_exe, "-i", file_path]
        process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, universal_newlines=True)
        _, stderr = process.communicate()
        
        # Regex for "Video: ..., 1234x5678, ..."
        match = re.search(r", (\d+)x(\d+)", stderr)
        if not match:
            # Fallback regex
            match = re.search(r"Video:.* (\d{3,})x(\d{3,})", stderr)
            
        if match:
            return int(match.group(1)), int(match.group(2))
    except:
        pass
    return None, None

def process_single_file(ffmpeg_exe, src_path, temp_raw_path):
    """
    Main logic to process one image. Returns True on success.
    """
    filename = os.path.basename(src_path)
    folder = os.path.dirname(src_path)
    name = os.path.splitext(filename)[0]
    output_video = os.path.join(folder, f"Video_{name}_{int(DURATION)}s.mp4")

    print(f"\nProcessing: {filename}...")

    try:
        # 1. Probe Dimensions
        w, h = get_image_dims(ffmpeg_exe, src_path)
        if w is None:
            print("  [SKIP] Could not detect dimensions (Format might be unsupported).")
            return False
        
        print(f"  Dims: {w} x {h}")

        # 2. Convert to Raw (Save to SSD)
        if os.path.exists(temp_raw_path):
            try: os.remove(temp_raw_path)
            except: pass

        print(f"  Streaming to raw temp file (SSD)...")
        # FFmpeg reads CR3/ARW/DNG automatically if the codec is supported
        cmd_convert = [
            ffmpeg_exe, "-y",
            "-i", src_path,
            "-f", "rawvideo",
            "-pix_fmt", "rgb24",
            temp_raw_path
        ]
        subprocess.run(cmd_convert, check=True, stderr=subprocess.DEVNULL)

        # 3. Memory Map & Render
        raw_map = np.memmap(temp_raw_path, dtype='uint8', mode='r', shape=(h, w, 3))
        
        # Geometry
        crop_h = h
        target_aspect = VIDEO_WIDTH / VIDEO_HEIGHT
        crop_w = int(crop_h * target_aspect)
        
        if crop_w > w:
            print(f"  [SKIP] Image too narrow (Needs {crop_w}px, has {w}px).")
            del raw_map
            return False

        # Video Writer
        fourcc = cv2.VideoWriter_fourcc(*'mp4v') 
        video = cv2.VideoWriter(output_video, fourcc, FPS, (VIDEO_WIDTH, VIDEO_HEIGHT))
        
        total_frames = int(DURATION * FPS)
        max_pan_x = w - crop_w
        step_per_frame = max_pan_x / total_frames
        
        # Render Loop
        for i in range(total_frames):
            curr_x = int(i * step_per_frame)
            
            # Read from SSD (Virtual RAM)
            frame_raw = raw_map[0:crop_h, curr_x:curr_x+crop_w]
            
            # Resize
            frame_final = cv2.resize(frame_raw, (VIDEO_WIDTH, VIDEO_HEIGHT), interpolation=cv2.INTER_LINEAR)
            
            # RGB -> BGR
            frame_bgr = cv2.cvtColor(frame_final, cv2.COLOR_RGB2BGR)
            video.write(frame_bgr)
            
            if i % 30 == 0 or i == total_frames - 1:
                percent = ((i + 1) / total_frames) * 100
                print(f"\r  Rendered {percent:.1f}% ({i+1}/{total_frames})", end='')

        video.release()
        del raw_map # Close file lock
        print(f"\n  [DONE] Saved video.")
        return True

    except Exception as e:
        print(f"\n  [ERROR] Failed: {e}")
        return False
    finally:
        if os.path.exists(temp_raw_path):
            try: os.remove(temp_raw_path)
            except: pass

def main():
    print("--- PANORAMA PANNER (UNIVERSAL BATCH) ---")
    
    ffmpeg_exe = imageio_ffmpeg.get_ffmpeg_exe()
    temp_dir = os.environ.get('TEMP', os.path.expanduser('~'))
    temp_raw_path = os.path.join(temp_dir, "pano_batch_temp.raw")
    
    # 1. Select Mode
    print("\nSelect Mode:")
    print("  [S] Single File")
    print("  [B] Batch Folder")
    mode = input("Choice (s/b): ").strip().lower()
    
    files = []
    root = tk.Tk()
    root.withdraw()

    if mode == 'b':
        folder = filedialog.askdirectory(title="Select Input Folder")
        if not folder: return
        
        # UPDATE: Added Raw formats (*.CR3, *.ARW, *.DNG) to this list
        valid_extensions = ['*.tif', '*.tiff', '*.jpg', '*.jpeg', '*.png', '*.bmp', '*.psd', '*.cr3', '*.arw', '*.dng']
        
        for ext in valid_extensions:
            files.extend(glob.glob(os.path.join(folder, ext)))
            files.extend(glob.glob(os.path.join(folder, ext.upper())))
            
        files = sorted(list(set(files)))
    else:
        f = filedialog.askopenfilename(title="Select Image")
        if f: files = [f]

    if not files: return

    # 2. Batch Loop
    print(f"\n--- STARTING BATCH ({len(files)} files) ---")
    
    success_count = 0
    start_batch = time.time()
    
    for i, f in enumerate(files):
        print(f"\n[File {i+1} of {len(files)}]")
        if process_single_file(ffmpeg_exe, f, temp_raw_path):
            success_count += 1
            
    elapsed = (time.time() - start_batch) / 60
    print(f"\n--- BATCH COMPLETE ---")
    print(f"Processed: {success_count}/{len(files)}")
    print(f"Total Time: {elapsed:.1f} minutes")
    
    input("Press Enter to exit...")

if __name__ == "__main__":
    main()

--- PANORAMA BATCH STITCHER STARTING ---
Please select your INPUT folder from the popup window...
No files found!
